# Detección de uso de casco de seguridad con YOLOv5

- Objetivo: detectar si los trabajadores en una imagen de obra o planta llevan puesto el casco de seguridad, usando un modelo de detección de objetos entrenado con transfer learning.
- Caso de uso: monitoreo automático de cumplimiento de EPP (equipo de protección personal) a partir de cámaras fijas o imágenes de campo.
- **Nota sobre credenciales:** este notebook usa una variable de entorno para la API key de Roboflow. Nunca se debe escribir una API key directamente en el código, ni subirla a un repositorio público.
- Artículo completo, con el razonamiento detrás de cada decisión: [fuzzyfrog.ai/es/ai-lab/proyectos/industria/deteccion-uso-casco-seguridad-yolov5](https://fuzzyfrog.ai/es/ai-lab/proyectos/industria/deteccion-uso-casco-seguridad-yolov5/)


## Diagrama del pipeline

- **Entrada:** imagen de obra o planta con una o más personas.
- **Modelo:** YOLOv5, con transfer learning desde pesos preentrenados en COCO.
- **Clases:** casco puesto y cabeza sin casco (se evaluó también incluir una clase "persona" completa, ver decisión clave en Construcción de la solución).
- **Salida:** cajas delimitadoras por cada casco o cabeza detectada, con su clase y confianza.


## Carga de datos

- El dataset usado es un conjunto público de imágenes de obra etiquetadas para detección de cascos de seguridad, alojado en Roboflow.
- Las anotaciones originales están en formato PASCAL VOC y se exportan a formato YOLOv5 antes de entrenar.
- **Nunca subas tu API key directamente en el notebook.** Usa una variable de entorno, como se muestra abajo.


In [ ]:
# Paso 1: clonar YOLOv5 e instalar dependencias
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
%pip install -qr requirements.txt

import torch
import os
from IPython.display import Image, clear_output

print(f"Configuracion completada. Usando torch {torch.__version__} "
      f"({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")


In [ ]:
# Configurar el directorio de datasets
os.environ["DATASET_DIRECTORY"] = "/content/datasets"

# La API key NUNCA debe escribirse en el codigo.
# En Colab: Herramientas > Secretos, o usar una variable de entorno.
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY")
if ROBOFLOW_API_KEY is None:
    raise ValueError(
        "Define la variable de entorno ROBOFLOW_API_KEY antes de continuar. "
        "En Colab: os.environ['ROBOFLOW_API_KEY'] = 'tu_key_aqui' en una celda que NO subas a git."
    )


In [ ]:
# Descargar el dataset publico de deteccion de cascos de seguridad desde Roboflow
!pip install -q roboflow
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("TU_WORKSPACE").project("hard-hat-detection")  # ajustar al dataset publico elegido
dataset = project.version(1).download("yolov5")


## Explicación de datos

- El dataset tiene imágenes de trabajadores en distintos ángulos, distancias y condiciones de luz — variabilidad real de cámaras de obra, no fotos de estudio.
- Las clases originales incluían "casco", "cabeza sin casco" y "persona". La clase "persona" tenía muy pocas instancias etiquetadas respecto a las otras dos.


## Análisis de datos

- Antes de entrenar, se revisó el balance de clases del dataset. La clase "persona" estaba muy subrepresentada frente a "casco" y "cabeza sin casco".
- Este desbalance es la base de la decisión clave documentada en la siguiente sección — no una nota al margen, es lo que definió la arquitectura de clases del modelo final.


## Modelado — entrenamiento con YOLOv5

### Decisión clave: eliminar la clase "persona"

Con pocas instancias etiquetadas, el modelo no aprendía a detectar personas de forma confiable, y ese bajo desempeño en una sola clase arrastraba hacia abajo el mAP promedio de las 3 clases. Se decidió entrenar solo con "casco" y "cabeza sin casco", que es exactamente la información que el caso de uso necesita — no hace falta detectar personas para verificar cumplimiento de EPP, basta con detectar cascos.


In [ ]:
# Prueba de humo del pipeline de entrenamiento con un dataset sintetico pequeno
# (Para el dataset real completo, usar dataset.location del paso de descarga de Roboflow)

# Ver dataset/data.yaml para el formato esperado:
# train: ruta/a/train/images
# val: ruta/a/val/images
# nc: 2
# names: ['casco', 'sin_casco']

!python train.py --img 128 --batch 8 --epochs 3 \
    --data ../dataset_sintetico/data.yaml --weights yolov5n.pt --cache --device cpu


**Nota sobre esta ejecución:** las 3 épocas y el dataset reducido de esta celda son una prueba de humo, para confirmar que el pipeline de entrenamiento ejecuta sin errores de principio a fin. No es una estimación de desempeño real. Los resultados reales, obtenidos entrenando 100 épocas con el dataset completo en una GPU de Colab, se documentan en la sección de Evaluación.


## Evaluación

- Con el dataset completo (imágenes reales de obra) y 100 épocas de entrenamiento en GPU, se obtuvieron los resultados reales de este proyecto.
- Se compararon dos configuraciones: con la clase "persona" incluida, y sin ella.


In [ ]:
# Inferencia sobre el set de prueba (o el set de validacion sintetico, en esta demo)
!python detect.py --weights runs/train/exp/weights/best.pt --img 416 --conf 0.1 \
    --source {dataset.location}/test/images


In [ ]:
import glob
from IPython.display import display

for i, imageName in enumerate(glob.glob('runs/detect/exp/*.jpg')):
    if i % 25 == 0:
        display(Image(filename=imageName))


### Resultado real, con el dataset completo (100 épocas, GPU)

| Configuración | mAP |
|---|---|
| Con clase "persona" incluida | ~0.61 |
| Sin clase "persona" (solo casco / sin casco) | **0.917** |

El salto de 0.61 a 0.917 no vino de ajustar hiperparámetros ni de una arquitectura distinta — vino de remover una clase mal representada en los datos. Es la misma lección que en otros proyectos de este laboratorio: el desbalance de clases pesa más en el resultado final que casi cualquier ajuste posterior al modelo.

**Limitación real, documentada sin maquillar:** el entrenamiento se detuvo en 100 épocas por las limitaciones de uso de GPU de Colab, no porque el modelo hubiera dejado de mejorar. Es probable que más épocas de entrenamiento hubieran subido el mAP todavía más.


## Hallazgos principales

- Remover una clase con pocas instancias etiquetadas ("persona") subió el mAP de ~0.61 a 0.917 — el balance de clases importó más que cualquier ajuste de arquitectura o hiperparámetros.
- El caso de uso real (verificar cumplimiento de casco) no necesitaba la clase "persona" para empezar — simplificar el problema al mínimo necesario mejoró tanto la simplicidad como el desempeño.
- Las limitaciones de cómputo (GPU de Colab) son una restricción real de estos proyectos, y documentarlas explícitamente es más honesto que presentar el resultado final como el techo del enfoque.
- Nunca subas una API key directamente en un notebook. Usa variables de entorno o el gestor de secretos de tu entorno de ejecución.
